# NammaMart data profiling and validation

This notebook reproduces the Part 0 checks from `profile_nammamart.py`: file structure, nulls, duplicates, ranges, categories, referential integrity, business rules, and incremental-feed behavior.

The source CSVs are read as strings first so malformed values are preserved and reported rather than silently coerced.

In [2]:
from pathlib import Path
import pandas as pd

# Resolve the repository root whether the notebook kernel starts in the repo or include/.
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'assignment').exists():
    REPO_ROOT = REPO_ROOT.parent
DATA_DIR = REPO_ROOT / 'include' / 'data' / 'nammamart'
OUTPUT_DIR = REPO_ROOT / 'assignment' / 'profile'
FILES = ['orders.csv', 'payments.csv', 'customers.csv', 'products.csv', 'stores.csv', 'orders_2026-10-01_increment.csv']

# Read as text so invalid statuses, malformed numbers, and blank values remain visible.
frames = {filename: pd.read_csv(DATA_DIR / filename, dtype=str, keep_default_na=False) for filename in FILES}
print(f'Repository: {REPO_ROOT}')
print(f'Data directory: {DATA_DIR}')
for filename, frame in frames.items():
    print(f'{filename}: {len(frame):,} rows x {len(frame.columns)} columns')

Repository: c:\Users\M720s_5\Rudin\FDE\Github\IIT_Roorkie_CEC_Futurenece_Data_Pipeline_to_ML_Model
Data directory: c:\Users\M720s_5\Rudin\FDE\Github\IIT_Roorkie_CEC_Futurenece_Data_Pipeline_to_ML_Model\include\data\nammamart
orders.csv: 2,412 rows x 13 columns
payments.csv: 2,228 rows x 7 columns
customers.csv: 405 rows x 10 columns
products.csv: 152 rows x 10 columns
stores.csv: 15 rows x 8 columns
orders_2026-10-01_increment.csv: 160 rows x 13 columns


In [3]:
# These metadata definitions drive the reusable profiling checks.
KEYS = {'orders.csv': 'order_id', 'payments.csv': 'payment_id', 'customers.csv': 'customer_id', 'products.csv': 'product_id', 'stores.csv': 'store_id', 'orders_2026-10-01_increment.csv': 'order_id'}
NUMERIC = {'orders.csv': ['quantity', 'unit_price', 'discount_pct', 'order_amount', 'delivery_minutes'], 'payments.csv': ['amount_paid'], 'customers.csv': ['pincode'], 'products.csv': ['mrp', 'cost_price'], 'stores.csv': ['pincode', 'capacity_orders_per_day'], 'orders_2026-10-01_increment.csv': ['quantity', 'unit_price', 'discount_pct', 'order_amount', 'delivery_minutes']}
DATES = {'orders.csv': ['order_ts'], 'payments.csv': ['payment_ts'], 'customers.csv': ['signup_date'], 'products.csv': ['launch_date'], 'stores.csv': ['open_date'], 'orders_2026-10-01_increment.csv': ['order_ts']}
CATEGORICAL = {'orders.csv': ['order_status', 'payment_mode', 'channel'], 'payments.csv': ['payment_mode', 'payment_status'], 'customers.csv': ['city', 'area', 'loyalty_tier', 'is_active'], 'products.csv': ['category', 'sub_category', 'brand', 'unit', 'is_perishable'], 'stores.csv': ['zone', 'area'], 'orders_2026-10-01_increment.csv': ['order_status', 'payment_mode', 'channel']}

# Build a long-form profile: row counts, blanks, duplicate keys, ranges, dates, and categories.
profile = []
for filename, frame in frames.items():
    profile.append({'file': filename, 'section': 'summary', 'column': '__file__', 'metric': 'row_count', 'value': len(frame)})
    for column in frame.columns:
        profile.append({'file': filename, 'section': 'nulls', 'column': column, 'metric': 'blank_count', 'value': int(frame[column].str.strip().eq('').sum())})
    key = KEYS[filename]
    duplicate_groups = frame[key].value_counts().loc[lambda values: values > 1]
    profile.append({'file': filename, 'section': 'duplicates', 'column': key, 'metric': 'duplicate_key_groups', 'value': len(duplicate_groups)})
    profile.append({'file': filename, 'section': 'duplicates', 'column': key, 'metric': 'duplicate_rows_beyond_first', 'value': int((duplicate_groups - 1).sum())})
    for column in NUMERIC.get(filename, []):
        values = pd.to_numeric(frame[column], errors='coerce').dropna()
        if not values.empty:
            profile.extend([{'file': filename, 'section': 'numeric_range', 'column': column, 'metric': 'min', 'value': values.min()}, {'file': filename, 'section': 'numeric_range', 'column': column, 'metric': 'max', 'value': values.max()}])
    for column in DATES.get(filename, []):
        values = pd.to_datetime(frame[column], errors='coerce').dropna()
        if not values.empty:
            profile.extend([{'file': filename, 'section': 'date_range', 'column': column, 'metric': 'min', 'value': values.min()}, {'file': filename, 'section': 'date_range', 'column': column, 'metric': 'max', 'value': values.max()}])
    for column in CATEGORICAL.get(filename, []):
        values = sorted(value for value in frame[column].unique() if value.strip())
        profile.append({'file': filename, 'section': 'distinct_values', 'column': column, 'metric': 'values', 'value': ' | '.join(values)})

profile_df = pd.DataFrame(profile)
display(profile_df[profile_df.metric == 'row_count'])
blank_counts = pd.to_numeric(profile_df.loc[profile_df.metric == 'blank_count', 'value'], errors='coerce')
display(profile_df.loc[(profile_df.metric == 'blank_count') & blank_counts.gt(0)])
display(profile_df[profile_df.section == 'duplicates'])

,file,section,column,metric,value
0,orders.csv,summary,__file__,row_count,2412
31,payments.csv,summary,__file__,row_count,2228
47,customers.csv,summary,__file__,row_count,405
68,products.csv,summary,__file__,row_count,152
92,stores.csv,summary,__file__,row_count,15
111,orders_2026-10-01_increment.csv,summary,__file__,row_count,160


,file,section,column,metric,value
2,orders.csv,nulls,order_ts,blank_count,1
3,orders.csv,nulls,customer_id,blank_count,18
13,orders.csv,nulls,delivery_minutes,blank_count,164
38,payments.csv,nulls,gateway_ref,blank_count,301
52,customers.csv,nulls,city,blank_count,6
71,products.csv,nulls,category,blank_count,3
95,stores.csv,nulls,zone,blank_count,1


,file,section,column,metric,value
14,orders.csv,duplicates,order_id,duplicate_key_groups,12
15,orders.csv,duplicates,order_id,duplicate_rows_beyond_first,12
39,payments.csv,duplicates,payment_id,duplicate_key_groups,4
40,payments.csv,duplicates,payment_id,duplicate_rows_beyond_first,4
58,customers.csv,duplicates,customer_id,duplicate_key_groups,5
59,customers.csv,duplicates,customer_id,duplicate_rows_beyond_first,5
79,products.csv,duplicates,product_id,duplicate_key_groups,2
80,products.csv,duplicates,product_id,duplicate_rows_beyond_first,2
101,stores.csv,duplicates,store_id,duplicate_key_groups,0
102,stores.csv,duplicates,store_id,duplicate_rows_beyond_first,0


In [4]:
# Validation helpers record one issue per failed rule with the assignment's required fields.
ALLOWED_STATUS = {'DELIVERED', 'CANCELLED', 'RETURNED'}
ALLOWED_MODES = {'UPI', 'Card', 'Cash on Delivery', 'Wallet'}
issues = []
def add_issue(file, row_key, column, problem, dimension, rule, severity):
    issues.append({'file': file, 'row_key': row_key, 'column': column, 'problem': problem, 'quality_dimension': dimension, 'proposed_rule': rule, 'severity': severity})

orders = frames['orders.csv']; payments = frames['payments.csv']; customers = frames['customers.csv']; products = frames['products.csv']; stores = frames['stores.csv']
order_ids = set(orders.order_id); customer_ids = set(customers.customer_id); product_ids = set(products.product_id); store_ids = set(stores.store_id)

# Primary-key checks apply to every source file.
for filename, frame in frames.items():
    key = KEYS[filename]
    for value, count in frame[key].value_counts().items():
        if count > 1:
            add_issue(filename, value, key, f'Duplicate key appears {count} times', 'Uniqueness', 'Primary key must be unique', 'Critical')

# Orders: completeness, foreign keys, allowed values, formula accuracy, timestamps, and delivery SLA.
for _, row in orders.iterrows():
    key = row.order_id
    for column, valid_keys in [('customer_id', customer_ids), ('product_id', product_ids), ('store_id', store_ids)]:
        if not row[column].strip():
            add_issue('orders.csv', key, column, f'{column} is blank', 'Completeness', f'Order {column} is required', 'Critical')
        elif row[column] not in valid_keys:
            add_issue('orders.csv', key, column, f'{column} does not exist', 'Consistency', f'Order {column} must exist in its master table', 'Critical')
    status = row.order_status.strip().upper(); mode = row.payment_mode.strip()
    if status not in ALLOWED_STATUS: add_issue('orders.csv', key, 'order_status', f'Invalid status: {row.order_status}', 'Validity', 'Status must be DELIVERED, CANCELLED or RETURNED', 'Critical')
    if mode not in ALLOWED_MODES: add_issue('orders.csv', key, 'payment_mode', f'Invalid payment mode: {row.payment_mode}', 'Validity', 'Payment mode must be one of the allowed values', 'Critical')
    quantity, unit, discount, amount = [pd.to_numeric(row[column], errors='coerce') for column in ['quantity', 'unit_price', 'discount_pct', 'order_amount']]
    if pd.isna(quantity) or quantity <= 0: add_issue('orders.csv', key, 'quantity', 'Quantity is missing or not greater than zero', 'Accuracy', 'quantity must be > 0', 'Critical')
    if pd.notna(discount) and not 0 <= discount <= 100: add_issue('orders.csv', key, 'discount_pct', 'Discount is outside 0-100', 'Validity', 'discount_pct must be between 0 and 100', 'Critical')
    if all(pd.notna(value) for value in [quantity, unit, discount, amount]) and abs(amount - round(quantity * unit * (1 - discount / 100), 2)) > 0.01: add_issue('orders.csv', key, 'order_amount', 'Order amount does not match the formula', 'Accuracy', 'order_amount = quantity * unit_price * (1 - discount_pct/100)', 'Critical')
    timestamp = pd.to_datetime(row.order_ts, errors='coerce')
    if pd.isna(timestamp): add_issue('orders.csv', key, 'order_ts', 'Timestamp cannot be parsed', 'Validity', 'order_ts must be a valid timestamp', 'Critical')
    elif timestamp > pd.Timestamp('2026-10-09 23:59:59'): add_issue('orders.csv', key, 'order_ts', 'Future-dated order timestamp', 'Timeliness', 'Order timestamp must not be future dated', 'Critical')
    delivery = pd.to_numeric(row.delivery_minutes, errors='coerce')
    if status == 'DELIVERED' and pd.isna(delivery): add_issue('orders.csv', key, 'delivery_minutes', 'Delivered order has no delivery time', 'Completeness', 'DELIVERED orders require delivery_minutes', 'Critical')
    if pd.notna(delivery) and delivery >= 120: add_issue('orders.csv', key, 'delivery_minutes', f'Implausible delivery time: {delivery}', 'Accuracy', 'Delivery time must be below 120 minutes', 'Critical')

# Payments: relationship and reconciliation checks.
order_amounts = orders.set_index('order_id').order_amount
for _, row in payments.iterrows():
    key = row.payment_id
    if row.payment_mode.strip() not in ALLOWED_MODES: add_issue('payments.csv', key, 'payment_mode', f'Invalid payment mode: {row.payment_mode}', 'Validity', 'Payment mode must be one of the allowed values', 'Critical')
    if row.order_id not in order_ids: add_issue('payments.csv', key, 'order_id', 'Payment references an unknown order', 'Consistency', 'Payment order_id must exist in orders', 'Critical')
    paid = pd.to_numeric(row.amount_paid, errors='coerce')
    expected_values = pd.to_numeric(orders.loc[orders.order_id == row.order_id, 'order_amount'], errors='coerce')
    expected = expected_values.iloc[0] if not expected_values.empty else pd.NA
    if row.payment_status.strip().upper() == 'SUCCESS' and pd.notna(paid) and pd.notna(expected) and abs(paid - expected) > 0.01: add_issue('payments.csv', key, 'amount_paid', 'SUCCESS payment does not equal order_amount', 'Consistency', 'A SUCCESS payment must equal its order amount', 'Critical')

# Validate master-data formats and payment timestamps.
for _, row in payments.iterrows():
    if pd.isna(pd.to_datetime(row.payment_ts, errors='coerce')): add_issue('payments.csv', row.payment_id, 'payment_ts', 'Timestamp cannot be parsed', 'Validity', 'payment_ts must be a valid timestamp', 'Critical')
for _, row in customers.iterrows():
    if not pd.Series([row.email]).str.match(r'^[^@\s]+@[^@\s]+\.[^@\s]+$').iloc[0]: add_issue('customers.csv', row.customer_id, 'email', 'Email does not match expected format', 'Validity', 'Email should match a basic email pattern', 'Warning')
    if not str(row.phone).isdigit() or len(str(row.phone)) != 10: add_issue('customers.csv', row.customer_id, 'phone', 'Phone is not exactly 10 digits', 'Validity', 'Phone should contain 10 digits', 'Warning')
    if not str(row.pincode).isdigit() or len(str(row.pincode)) != 6: add_issue('customers.csv', row.customer_id, 'pincode', 'Pincode is not exactly 6 digits', 'Validity', 'Pincode should contain 6 digits', 'Warning')
    signup = pd.to_datetime(row.signup_date, errors='coerce')
    if pd.notna(signup) and signup > pd.Timestamp('2026-10-09'): add_issue('customers.csv', row.customer_id, 'signup_date', 'Future-dated signup date', 'Timeliness', 'Signup date must not be future dated', 'Critical')
for _, row in products.iterrows():
    mrp = pd.to_numeric(row.mrp, errors='coerce'); cost = pd.to_numeric(row.cost_price, errors='coerce')
    if not row.category.strip(): add_issue('products.csv', row.product_id, 'category', 'Category is blank', 'Completeness', 'Product category is required', 'Critical')
    if pd.isna(mrp) or mrp <= 0: add_issue('products.csv', row.product_id, 'mrp', 'MRP is missing or not positive', 'Accuracy', 'mrp must be > 0', 'Critical')
    if pd.notna(mrp) and pd.notna(cost) and cost > mrp: add_issue('products.csv', row.product_id, 'cost_price', 'Cost price exceeds MRP', 'Accuracy', 'cost_price must be <= mrp', 'Critical')
for _, row in stores.iterrows():
    if not row.zone.strip(): add_issue('stores.csv', row.store_id, 'zone', 'Store zone is blank', 'Completeness', 'Store zone is required for zone reporting', 'Critical')

# Incremental-feed check: existing IDs should be upserted, not appended.
increment = frames['orders_2026-10-01_increment.csv']
existing_increment = increment[increment.order_id.isin(order_ids)]
for order_id in existing_increment.order_id: add_issue('orders_2026-10-01_increment.csv', order_id, 'order_id', 'Existing order is an intentional correction/upsert candidate', 'Consistency', 'Merge existing order_id values during incremental load', 'Warning')

issues_df = pd.DataFrame(issues)
display(issues_df.groupby(['file', 'quality_dimension', 'severity']).size().reset_index(name='failed_rows'))
display(issues_df.head(20))

,file,quality_dimension,severity,failed_rows
0,customers.csv,Timeliness,Critical,3
1,customers.csv,Uniqueness,Critical,5
2,customers.csv,Validity,Warning,22
3,orders.csv,Accuracy,Critical,34
4,orders.csv,Completeness,Critical,26
5,orders.csv,Consistency,Critical,14
6,orders.csv,Timeliness,Critical,7
7,orders.csv,Uniqueness,Critical,12
8,orders.csv,Validity,Critical,20
9,orders_2026-10-01_increment.csv,Consistency,Warning,10


,file,row_key,column,problem,quality_dimension,proposed_rule,severity
0,orders.csv,ORD001462,order_id,Duplicate key appears 2 times,Uniqueness,Primary key must be unique,Critical
1,orders.csv,ORD001326,order_id,Duplicate key appears 2 times,Uniqueness,Primary key must be unique,Critical
2,orders.csv,ORD001434,order_id,Duplicate key appears 2 times,Uniqueness,Primary key must be unique,Critical
3,orders.csv,ORD000783,order_id,Duplicate key appears 2 times,Uniqueness,Primary key must be unique,Critical
4,orders.csv,ORD001239,order_id,Duplicate key appears 2 times,Uniqueness,Primary key must be unique,Critical
5,orders.csv,ORD001692,order_id,Duplicate key appears 2 times,Uniqueness,Primary key must be unique,Critical
6,orders.csv,ORD001870,order_id,Duplicate key appears 2 times,Uniqueness,Primary key must be unique,Critical
7,orders.csv,ORD001441,order_id,Duplicate key appears 2 times,Uniqueness,Primary key must be unique,Critical
8,orders.csv,ORD002129,order_id,Duplicate key appears 2 times,Uniqueness,Primary key must be unique,Critical
9,orders.csv,ORD001549,order_id,Duplicate key appears 2 times,Uniqueness,Primary key must be unique,Critical


In [5]:
# Persist the same artifacts used by the Python profiler and show the main totals.
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
profile_df.to_csv(OUTPUT_DIR / 'profile_summary_from_notebook.csv', index=False)
issues_df.to_csv(OUTPUT_DIR / 'data_issues_log_from_notebook.csv', index=False)
print(f'Profile rows: {len(profile_df):,}')
print(f'Validation issues: {len(issues_df):,}')
print(f'Increment rows: {len(increment):,}; existing order IDs: {len(existing_increment):,}; new order IDs: {len(increment) - len(existing_increment):,}')

Profile rows: 142
Validation issues: 199
Increment rows: 160; existing order IDs: 10; new order IDs: 150
